In [ ]:
pip install tensorflow


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Flatten, concatenate
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import numpy as np
import pickle

# Step 1: Load your data
data = pd.read_csv('./cleaned_tweets.csv')

# Step 2: Preprocess the data

# Ensure text is a string (if any non-string data exists)
data['text'] = data['text'].astype(str)

# Check for missing values and handle them if needed
data = data.dropna(subset=['text', 'user_name', 'tone'])  # Drop rows with missing values in relevant columns

# Step 3: Encode user-related data (user_name column) using LabelEncoder
label_encoder = LabelEncoder()
data['user_encoded'] = label_encoder.fit_transform(data['user_name'])

# Step 4: Encode the tone column (this will be the target variable)
tone_encoder = LabelEncoder()
data['tone_encoded'] = tone_encoder.fit_transform(data['tone'])  # Map tone to numeric values (e.g., positive=0, negative=1, neutral=2)

# Step 5: Prepare text data for the model

# Tokenize the text
tokenizer = Tokenizer(num_words=5000)  # You can increase num_words if needed
tokenizer.fit_on_texts(data['text'])

# Convert text to sequences of tokens
X_text = tokenizer.texts_to_sequences(data['text'])

# Pad sequences to have equal length
max_sequence_length = max([len(sequence) for sequence in X_text])
X_text_padded = pad_sequences(X_text, maxlen=max_sequence_length)

# Step 6: Prepare user-related data (user_encoded) as an additional feature
X_user = data['user_encoded'].values

# Step 7: Prepare the target variable (tone_encoded)
y = data['tone_encoded']  # 'tone_encoded' is the target (sentiment)

# Step 8: Split the data into training and testing sets
X_train_text, X_test_text, X_train_user, X_test_user, y_train, y_test = train_test_split(
    X_text_padded, X_user, y, test_size=0.2, random_state=42
)

# Step 9: Build the model

# Define the input layers
text_input = Input(shape=(max_sequence_length,), name='text_input')
user_input = Input(shape=(1,), name='user_input')

# Embedding and LSTM layers for text input
embedding = Embedding(input_dim=5000, output_dim=128)(text_input)
lstm = LSTM(128, dropout=0.2, recurrent_dropout=0.2)(embedding)

# Flatten and combine with user input
flatten = Flatten()(lstm)
combined = concatenate([flatten, user_input])

# Dense layer and output layer for sentiment classification
dense = Dense(128, activation='relu')(combined)
output = Dense(len(tone_encoder.classes_), activation='softmax')(dense)  # Using softmax for multi-class classification

# Step 10: Compile the model
model = Model(inputs=[text_input, user_input], outputs=output)
model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Step 11: Train the model
model.fit([X_train_text, X_train_user], y_train, epochs=10, batch_size=64, validation_data=([X_test_text, X_test_user], y_test))

# Step 12: Evaluate the model
loss, accuracy = model.evaluate([X_test_text, X_test_user], y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Save the model
model.save('sentiment_model.h5')  # Save the trained model

# Save the tokenizer
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# Save the label encoders
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

with open('tone_encoder.pkl', 'wb') as f:
    pickle.dump(tone_encoder, f)

print("Model, Tokenizer, and Encoders have been saved successfully.")


In [ ]:
# Step 1: Make predictions on the test set
y_pred_prob = model.predict([X_test_text, X_test_user])  # Predict the probabilities for each class

# Step 2: Convert probabilities to predicted class labels (for multi-class classification)
y_pred = np.argmax(y_pred_prob, axis=1)  # Get the index of the maximum probability as the predicted label

# Step 3: Decode the predicted and true labels (optional, for easier interpretation)
predicted_labels = tone_encoder.inverse_transform(y_pred)  # Convert numerical predictions back to tone labels
true_labels = tone_encoder.inverse_transform(y_test)  # Convert true labels back to tone labels

# Step 4: Evaluate the accuracy on the test set
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Calculate the accuracy
accuracy = accuracy_score(true_labels, predicted_labels)
print(f"Test Accuracy: {accuracy}")

# Step 5: Detailed classification report
print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels))

# Step 6: Confusion Matrix
print("\nConfusion Matrix:")
conf_matrix = confusion_matrix(true_labels, predicted_labels)
print(conf_matrix)

# Step 7: Visualize the confusion matrix (optional)
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=tone_encoder.classes_, yticklabels=tone_encoder.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# Step 8: Test with a single sample (optional)
# Example: Predict the sentiment for a new tweet
new_text = "I love this product, it's amazing!"  # Example tweet
new_user = label_encoder.transform(["user_name_example"])  # Replace with a real username from the dataset

# Tokenize and pad the new text
new_text_seq = tokenizer.texts_to_sequences([new_text])
new_text_padded = pad_sequences(new_text_seq, maxlen=max_sequence_length)

# Make a prediction for the new text
new_pred_prob = model.predict([new_text_padded, new_user])
new_pred = np.argmax(new_pred_prob, axis=1)
new_pred_label = tone_encoder.inverse_transform(new_pred)

print(f"Predicted Sentiment: {new_pred_label[0]}")  # Print predicted sentiment (e.g., positive, negative, neutral)


In [ ]:
print(data.isnull().sum())